In [1]:
import os
import pandas as pd
import numpy as np

In [2]:
# Load all parquet files in a directory
def load_parquets(dir):
    tables = {}
    for fname in os.listdir(dir):
        if fname.endswith(".parquet"):
            symbol = fname.replace(".parquet", "")
            df = pd.read_parquet(os.path.join(dir, fname))
            tables[symbol] = df
    return tables

def show_best_params(table):
    # Best overall
    best_all = table.loc[table["score"].idxmax()]

    # Best excluding k = 0
    table_nonzero_k = table.loc[table.index.get_level_values("k") != 0]
    best_nonzero_k = table_nonzero_k.loc[table_nonzero_k["score"].idxmax()]

    print("=== BEST OVERALL ===")
    print(best_all)

    print("\n=== BEST WITH k != 0 ===")
    print(best_nonzero_k)

def show_results(table):
    display(table)

In [3]:
param_tables = load_parquets("params/")
result_tables = load_parquets("results/")

In [4]:
# Show best validation parameter and test result of a stock symbol
symbol = "BBCA"
try:
    show_best_params(param_tables[symbol])
except:
    print(f"No available tuned parameters for the symbol {symbol}")
try:
    show_results(result_tables[symbol])
except:
    print(f"No available test results for the symbol {symbol}")

=== BEST OVERALL ===
mean_excess    0.094667
std_excess     0.021221
score          0.073446
Name: (GaussianNB, 7, 0), dtype: float64

=== BEST WITH k != 0 ===
mean_excess    0.097600
std_excess     0.029362
score          0.068238
Name: (GaussianNB, 7, 3), dtype: float64


,Buy and Hold,MA Based,Breakout Based,Classifier Model,Mean Hybrid Model,Excess MA,Excess Breakout,Excess Classifier,Excess Hybrid,Excess Hybrid Std,Number of Samples
0,-0.16456,-0.018952,-0.115759,-0.118121,-0.05506,0.145608,0.048801,0.046439,0.109499,0.004811,15


In [5]:
# Analyze aggregated test performance across the 20 stocks that achieved positive validation scores
symbols = ["BBCA", "BYAN", "BBRI", "BMRI", 
           "ASII", "ICBP", "AMRT", "UNTR",
           "BRPT", "ISAT", "CPIN", "UNVR", "KLBF",
           "MBMA", "ADMR", "BRMS",
           "MEGA", "TBIG", "BNGA", "MDKA"]

n = len(symbols)

buy_and_hold = np.zeros(n, dtype = np.float32)
excess_ma = np.zeros(n, dtype = np.float32)
excess_breakout = np.zeros(n, dtype = np.float32)
excess_classifier = np.zeros(n, dtype = np.float32)
excess_hybrid = np.zeros(n, dtype = np.float32)
hybrid_score = np.zeros(n, dtype = np.float32)

for i in range(n):
    table = result_tables[symbols[i]].loc[0]
    buy_and_hold[i] = table["Buy and Hold"]
    excess_ma[i] = table['Excess MA']
    excess_breakout[i] = table['Excess Breakout']
    excess_classifier[i] = table['Excess Classifier']
    excess_hybrid[i] = table['Excess Hybrid']
    hybrid_score[i] = table['Excess Hybrid'] - table['Excess Hybrid Std']

returns = {
    'Mean Buy and Hold': buy_and_hold.mean(),
    'Mean Excess MA': excess_ma.mean(),
    'Mean Excess Breakout': excess_breakout.mean(),
    'Mean Excess Classifier': excess_classifier.mean(),
    'Mean Excess Hybrid': excess_hybrid.mean(),
    'Mean Hybrid Score': hybrid_score.mean()
}

ma_returns = excess_ma + buy_and_hold
breakout_returns = excess_breakout + buy_and_hold
classifier_returns = excess_classifier + buy_and_hold
hybrid_returns = excess_hybrid + buy_and_hold

sharpees = {
    'Buy and Hold Sharpee': buy_and_hold.mean() / buy_and_hold.std(),
    'MA Sharpee': ma_returns.mean() / ma_returns.std(),
    'Breakout Sharpee': breakout_returns.mean() / breakout_returns.std(),
    'Classifier Sharpee': classifier_returns.mean() / classifier_returns.std(),
    'Hybrid Sharpee': hybrid_returns.mean() / hybrid_returns.std(),   
}

np.set_printoptions(formatter={'float': '{: 0.3f}'.format})

returns_df = pd.DataFrame([returns])
sharpees_df = pd.DataFrame([sharpees])
display(returns_df)
display(sharpees_df)

,Mean Buy and Hold,Mean Excess MA,Mean Excess Breakout,Mean Excess Classifier,Mean Excess Hybrid,Mean Hybrid Score
0,0.288032,-0.234528,-0.082404,-0.058212,-0.055524,-0.19253


,Buy and Hold Sharpee,MA Sharpee,Breakout Sharpee,Classifier Sharpee,Hybrid Sharpee
0,0.430128,0.18592,0.340737,0.479149,0.396448


In [ ]:
# Analyze performance of the hybrid model relative to the classifier other than mean returns
diff = excess_hybrid - excess_classifier

wins = diff > 0
losses = diff < 0

skew_table = {
    "Win rate": wins.mean(),
    "Loss rate": losses.mean(),

    "Average win": diff[wins].mean() if wins.any() else 0,
    "Average loss": diff[losses].mean() if losses.any() else 0,

    "Win/Loss magnitude ratio":
        abs(diff[wins].mean() / diff[losses].mean())
        if losses.any() else np.inf,

    "Mean improvement": diff.mean(),
    "Median improvement": np.median(diff),

    "Max win": diff.max(),
    "Max loss": diff.min(),

    "Skewness": pd.Series(diff).skew()
}

skew_table = pd.Series(skew_table)
print(skew_table)

Win rate                    0.350000
Loss rate                   0.650000
Average win                 0.225786
Average loss               -0.117441
Win/Loss magnitude ratio    1.922548
Mean improvement            0.002688
Median improvement         -0.010705
Max win                     0.730225
Max loss                   -0.519863
Skewness                    0.619145
dtype: float64
